### Imports

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer, AutoModel, pipeline
import string
from datetime import datetime

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download("punkt")
nltk.download("stopwords")

STOPWORDS = set(stopwords.words("english"))
vader = SentimentIntensityAnalyzer()

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\02_Florian_Benutzer\AppData\Roaming\nltk_data
[nltk_data]     ...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\02_Florian_Benutzer\AppData\Roaming\nltk_data
[nltk_data]     ...
[nltk_data]   Package stopwords is already up-to-date!


In [3]:
import re
import warnings

import numpy as np
import pandas as pd

from tqdm import tqdm

from sentence_transformers import SentenceTransformer
from transformers import pipeline

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

from bertopic import BERTopic

warnings.filterwarnings("ignore")

In [ ]:
# import pandas as pd
# import numpy as np
# import re

# from transformers import pipeline, AutoTokenizer, AutoModel
# import torch

# CSV-Daten laden

Donald Trump Tweets bis 2021

In [2]:
tweets = pd.read_csv("C:/Users/02_Florian_Benutzer/Desktop/trump_oil_analysis_v2/00_data/trump_tweets.csv")

In [3]:
tweets.head()

,id,text,is_retweet,is_deleted,device,favorites,retweets,datetime,is_flagged,date
0,9.845497e+16,Republicans and Democrats have both created ou...,False,False,TweetDeck,49,255,2011-08-02T18:07:48Z,False,2011-08-02
1,1.234653e+18,I was thrilled to be back in the Great city of...,False,False,Twitter for iPhone,73748,17404,2020-03-03T01:34:50Z,False,2020-03-03
2,1.218011e+18,RT @CBS_Herridge: READ: Letter to surveillance...,True,False,Twitter for iPhone,0,7396,2020-01-17T03:22:47Z,False,2020-01-17
3,1.304875e+18,The Unsolicited Mail In Ballot Scam is a major...,False,False,Twitter for iPhone,80527,23502,2020-09-12T20:10:58Z,False,2020-09-12
4,1.218160e+18,RT @MZHemingway: Very friendly telling of even...,True,False,Twitter for iPhone,0,9081,2020-01-17T13:13:59Z,False,2020-01-17


In [4]:
tweets.shape

(56571, 10)

In [5]:
tweets_v1 = tweets.copy()

In [6]:
# 2. Den String in ein "naives" (zeitzonenloses) Datetime-Objekt umwandeln
# (Pandas erkennt das Format meistens automatisch)
tweets_v1['timestamp_utc'] = pd.to_datetime(tweets_v1['datetime'])

# 4. In UTC umrechnen
#tweets_v1['timestamp_utc'] = tweets_v1['timestamp'].dt.tz_localize('UTC')

# 5. (Optional) Wenn du das "+00:00" am Ende der UTC-Zeit nicht in der CSV haben willst,
# kannst du die Zeitzonen-Information wieder entfernen (die Zeit bleibt die von UTC):
#tweets_v1['timestamp_utc_clean'] = tweets_v1['timestamp_utc'].dt.tz_localize(None)

# Ergebnis überprüfen
#print(df[['Timestamp', 'Timestamp_UTC_clean']].head())

In [7]:
tweets_v1.dtypes

id                           float64
text                             str
is_retweet                      bool
is_deleted                      bool
device                           str
favorites                      int64
retweets                       int64
datetime                         str
is_flagged                      bool
date                             str
timestamp_utc    datetime64[us, UTC]
dtype: object

In [8]:
tweets_v1.shape

(56571, 11)

In [9]:
tweets_v1.head()

,id,text,is_retweet,is_deleted,device,favorites,retweets,datetime,is_flagged,date,timestamp_utc
0,9.845497e+16,Republicans and Democrats have both created ou...,False,False,TweetDeck,49,255,2011-08-02T18:07:48Z,False,2011-08-02,2011-08-02 18:07:48+00:00
1,1.234653e+18,I was thrilled to be back in the Great city of...,False,False,Twitter for iPhone,73748,17404,2020-03-03T01:34:50Z,False,2020-03-03,2020-03-03 01:34:50+00:00
2,1.218011e+18,RT @CBS_Herridge: READ: Letter to surveillance...,True,False,Twitter for iPhone,0,7396,2020-01-17T03:22:47Z,False,2020-01-17,2020-01-17 03:22:47+00:00
3,1.304875e+18,The Unsolicited Mail In Ballot Scam is a major...,False,False,Twitter for iPhone,80527,23502,2020-09-12T20:10:58Z,False,2020-09-12,2020-09-12 20:10:58+00:00
4,1.218160e+18,RT @MZHemingway: Very friendly telling of even...,True,False,Twitter for iPhone,0,9081,2020-01-17T13:13:59Z,False,2020-01-17,2020-01-17 13:13:59+00:00


Retweets werden aus der Analyse ausgeschlossen

In [10]:
tweets_v2 = tweets_v1.copy()

In [11]:
tweets_v2 = tweets_v2[tweets_v2['is_retweet'] == False] 

In [12]:
tweets_v2.shape

(46694, 11)

Die Spalten 'is_deleted', 'favorites', 'retweets' und 'is_flagged' werden entfernt um Data Leakage zu vermeiden

In [13]:
tweets_v2 = tweets_v2.drop(columns=['is_retweet', 'is_deleted', 'device', 'favorites', 'retweets', 'is_flagged'])

In [14]:
tweets_v2.shape

(46694, 5)

In [15]:
tweets_v2

,id,text,datetime,date,timestamp_utc
0,9.845497e+16,Republicans and Democrats have both created ou...,2011-08-02T18:07:48Z,2011-08-02,2011-08-02 18:07:48+00:00
1,1.234653e+18,I was thrilled to be back in the Great city of...,2020-03-03T01:34:50Z,2020-03-03,2020-03-03 01:34:50+00:00
3,1.304875e+18,The Unsolicited Mail In Ballot Scam is a major...,2020-09-12T20:10:58Z,2020-09-12,2020-09-12 20:10:58+00:00
6,1.223641e+18,Getting a little exercise this morning! https:...,2020-02-01T16:14:02Z,2020-02-01,2020-02-01 16:14:02+00:00
7,1.319502e+18,https://t.co/4qwCKQOiOw,2020-10-23T04:52:14Z,2020-10-23,2020-10-23 04:52:14+00:00
...,...,...,...,...,...
56555,1.213079e+18,"Iran never won a war, but never lost a negotia...",2020-01-03T12:44:30Z,2020-01-03,2020-01-03 12:44:30+00:00
56559,1.212177e+18,Thank you to the @dcexaminer Washington Examin...,2020-01-01T01:03:15Z,2020-01-01,2020-01-01 01:03:15+00:00
56560,1.212175e+18,One of my greatest honors was to have gotten C...,2020-01-01T00:55:01Z,2020-01-01,2020-01-01 00:55:01+00:00
56569,1.319384e+18,Just signed an order to support the workers of...,2020-10-22T21:04:21Z,2020-10-22,2020-10-22 21:04:21+00:00


In [16]:
tweets_v2.to_csv(r"C:\Users\02_Florian_Benutzer\Desktop\trump_oil_analysis_v2\00_data\tweets_v2.csv", sep=",", index=False, encoding="utf-8")

# Feature Engineering

### 1. Imports

In [56]:
df = pd.read_csv(r"C:\Users\02_Florian_Benutzer\Desktop\trump_oil_analysis_v2\00_data\tweets_v2.csv")

In [57]:
df

,id,text,datetime,date,timestamp_utc
0,9.845497e+16,Republicans and Democrats have both created ou...,2011-08-02T18:07:48Z,2011-08-02,2011-08-02 18:07:48+00:00
1,1.234653e+18,I was thrilled to be back in the Great city of...,2020-03-03T01:34:50Z,2020-03-03,2020-03-03 01:34:50+00:00
2,1.304875e+18,The Unsolicited Mail In Ballot Scam is a major...,2020-09-12T20:10:58Z,2020-09-12,2020-09-12 20:10:58+00:00
3,1.223641e+18,Getting a little exercise this morning! https:...,2020-02-01T16:14:02Z,2020-02-01,2020-02-01 16:14:02+00:00
4,1.319502e+18,https://t.co/4qwCKQOiOw,2020-10-23T04:52:14Z,2020-10-23,2020-10-23 04:52:14+00:00
...,...,...,...,...,...
46689,1.213079e+18,"Iran never won a war, but never lost a negotia...",2020-01-03T12:44:30Z,2020-01-03,2020-01-03 12:44:30+00:00
46690,1.212177e+18,Thank you to the @dcexaminer Washington Examin...,2020-01-01T01:03:15Z,2020-01-01,2020-01-01 01:03:15+00:00
46691,1.212175e+18,One of my greatest honors was to have gotten C...,2020-01-01T00:55:01Z,2020-01-01,2020-01-01 00:55:01+00:00
46692,1.319384e+18,Just signed an order to support the workers of...,2020-10-22T21:04:21Z,2020-10-22,2020-10-22 21:04:21+00:00


In [58]:
df.shape

(46694, 5)

### 2. Configuration

In [13]:
TEXT_COL = "text"
TIMESTAMP_COL = "timestamp_utc"

SENTENCE_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

PCA_COMPONENTS = 20
N_CLUSTERS = 10

TEST_SIZE = 0.20

### 3. Input

In [60]:
tweets_df = df.copy()

tweets_df[TIMESTAMP_COL] = pd.to_datetime(
    tweets_df[TIMESTAMP_COL],
    utc=True
)

tweets_df = tweets_df.sort_values(
    TIMESTAMP_COL
).reset_index(drop=True)

In [61]:
tweets_df

,id,text,datetime,date,timestamp_utc
0,1.698309e+09,Be sure to tune in and watch Donald Trump on L...,2009-05-04T18:54:25Z,2009-05-04,2009-05-04 18:54:25+00:00
1,1.701461e+09,Donald Trump will be appearing on The View tom...,2009-05-05T01:00:10Z,2009-05-05,2009-05-05 01:00:10+00:00
2,1.737480e+09,Donald Trump reads Top Ten Financial Tips on L...,2009-05-08T13:38:08Z,2009-05-08,2009-05-08 13:38:08+00:00
3,1.741161e+09,New Blog Post: Celebrity Apprentice Finale and...,2009-05-08T20:40:15Z,2009-05-08,2009-05-08 20:40:15+00:00
4,1.773561e+09,"""""""My persona will never be that of a wallflow...",2009-05-12T14:07:28Z,2009-05-12,2009-05-12 14:07:28+00:00
...,...,...,...,...,...
46689,1.346929e+18,https://t.co/Pm2PKV0Fp3,2021-01-06T21:17:24Z,2021-01-06,2021-01-06 21:17:24+00:00
46690,1.346955e+18,These are the things and events that happen wh...,2021-01-06T23:01:04Z,2021-01-06,2021-01-06 23:01:04+00:00
46691,1.347335e+18,https://t.co/csX07ZVWGe,2021-01-08T00:10:24Z,2021-01-08,2021-01-08 00:10:24+00:00
46692,1.347555e+18,"The 75,000,000 great American Patriots who vot...",2021-01-08T14:46:38Z,2021-01-08,2021-01-08 14:46:38+00:00


In [62]:
tweets_df.shape

(46694, 5)

### 4. Cleaning

In [63]:
def clean_text(text):
    if pd.isna(text):
        return ""

    text = str(text)

    text = re.sub(r"http\S+", "<url>", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [64]:
tweets_df["clean_text"] = (
    tweets_df[TEXT_COL]
    .fillna("")
    .apply(clean_text)
)

In [65]:
tweets_df

,id,text,datetime,date,timestamp_utc,clean_text
0,1.698309e+09,Be sure to tune in and watch Donald Trump on L...,2009-05-04T18:54:25Z,2009-05-04,2009-05-04 18:54:25+00:00,Be sure to tune in and watch Donald Trump on L...
1,1.701461e+09,Donald Trump will be appearing on The View tom...,2009-05-05T01:00:10Z,2009-05-05,2009-05-05 01:00:10+00:00,Donald Trump will be appearing on The View tom...
2,1.737480e+09,Donald Trump reads Top Ten Financial Tips on L...,2009-05-08T13:38:08Z,2009-05-08,2009-05-08 13:38:08+00:00,Donald Trump reads Top Ten Financial Tips on L...
3,1.741161e+09,New Blog Post: Celebrity Apprentice Finale and...,2009-05-08T20:40:15Z,2009-05-08,2009-05-08 20:40:15+00:00,New Blog Post: Celebrity Apprentice Finale and...
4,1.773561e+09,"""""""My persona will never be that of a wallflow...",2009-05-12T14:07:28Z,2009-05-12,2009-05-12 14:07:28+00:00,"""""""My persona will never be that of a wallflow..."
...,...,...,...,...,...,...
46689,1.346929e+18,https://t.co/Pm2PKV0Fp3,2021-01-06T21:17:24Z,2021-01-06,2021-01-06 21:17:24+00:00,<url>
46690,1.346955e+18,These are the things and events that happen wh...,2021-01-06T23:01:04Z,2021-01-06,2021-01-06 23:01:04+00:00,These are the things and events that happen wh...
46691,1.347335e+18,https://t.co/csX07ZVWGe,2021-01-08T00:10:24Z,2021-01-08,2021-01-08 00:10:24+00:00,<url>
46692,1.347555e+18,"The 75,000,000 great American Patriots who vot...",2021-01-08T14:46:38Z,2021-01-08,2021-01-08 14:46:38+00:00,"The 75,000,000 great American Patriots who vot..."


In [66]:
def is_useless_text(text):
    if pd.isna(text):
        return True

    text = str(text).strip()

    # entferne URLs, Mentions, Hashtags
    cleaned = re.sub(r"http\S+|@\w+|#\w+", "", text)

    # entferne extra whitespace
    cleaned = re.sub(r"\s+", "", cleaned)

    # wenn danach nichts übrig bleibt -> kein "normaler" Text
    return cleaned == ""

In [67]:
mask = tweets_df[TEXT_COL].apply(is_useless_text)

filtered_df = tweets_df[mask]

In [68]:
filtered_df

,id,text,datetime,date,timestamp_utc,clean_text
2861,2.488574e+17,@murekar http://t.co/czU4cRUC http://t.co/4dM...,2012-09-20T18:53:11Z,2012-09-20,2012-09-20 18:53:11+00:00,@murekar <url> <url>
2868,2.488783e+17,@sayhigreg http://t.co/yae18sRP,2012-09-20T20:16:36Z,2012-09-20,2012-09-20 20:16:36+00:00,@sayhigreg <url>
2939,2.510519e+17,@ForexBoxusd @LibertyU http://t.co/1H1iaeCd,2012-09-26T20:13:35Z,2012-09-26,2012-09-26 20:13:35+00:00,@ForexBoxusd @LibertyU <url>
4738,2.894677e+17,@aeroleila7 @CelebApprentice http://t.co/pdeN...,2013-01-10T20:24:09Z,2013-01-10,2013-01-10 20:24:09+00:00,@aeroleila7 @CelebApprentice <url>
5981,3.066699e+17,@JaylonSeales,2013-02-27T07:39:28Z,2013-02-27,2013-02-27 07:39:28+00:00,@JaylonSeales
...,...,...,...,...,...,...
46661,1.346304e+18,https://t.co/X0CIj5dFlV,2021-01-05T03:55:39Z,2021-01-05,2021-01-05 03:55:39+00:00,<url>
46662,1.346304e+18,https://t.co/tJkmEhSqY5,2021-01-05T03:55:47Z,2021-01-05,2021-01-05 03:55:47+00:00,<url>
46685,1.346892e+18,https://t.co/izItBeFE6G,2021-01-06T18:49:54Z,2021-01-06,2021-01-06 18:49:54+00:00,<url>
46689,1.346929e+18,https://t.co/Pm2PKV0Fp3,2021-01-06T21:17:24Z,2021-01-06,2021-01-06 21:17:24+00:00,<url>


In [69]:
tweets_df = tweets_df[~mask]

In [70]:
tweets_df

,id,text,datetime,date,timestamp_utc,clean_text
0,1.698309e+09,Be sure to tune in and watch Donald Trump on L...,2009-05-04T18:54:25Z,2009-05-04,2009-05-04 18:54:25+00:00,Be sure to tune in and watch Donald Trump on L...
1,1.701461e+09,Donald Trump will be appearing on The View tom...,2009-05-05T01:00:10Z,2009-05-05,2009-05-05 01:00:10+00:00,Donald Trump will be appearing on The View tom...
2,1.737480e+09,Donald Trump reads Top Ten Financial Tips on L...,2009-05-08T13:38:08Z,2009-05-08,2009-05-08 13:38:08+00:00,Donald Trump reads Top Ten Financial Tips on L...
3,1.741161e+09,New Blog Post: Celebrity Apprentice Finale and...,2009-05-08T20:40:15Z,2009-05-08,2009-05-08 20:40:15+00:00,New Blog Post: Celebrity Apprentice Finale and...
4,1.773561e+09,"""""""My persona will never be that of a wallflow...",2009-05-12T14:07:28Z,2009-05-12,2009-05-12 14:07:28+00:00,"""""""My persona will never be that of a wallflow..."
...,...,...,...,...,...,...
46687,1.346904e+18,Please support our Capitol Police and Law Enfo...,2021-01-06T19:38:58Z,2021-01-06,2021-01-06 19:38:58+00:00,Please support our Capitol Police and Law Enfo...
46688,1.346913e+18,I am asking for everyone at the U.S. Capitol t...,2021-01-06T20:13:26Z,2021-01-06,2021-01-06 20:13:26+00:00,I am asking for everyone at the U.S. Capitol t...
46690,1.346955e+18,These are the things and events that happen wh...,2021-01-06T23:01:04Z,2021-01-06,2021-01-06 23:01:04+00:00,These are the things and events that happen wh...
46692,1.347555e+18,"The 75,000,000 great American Patriots who vot...",2021-01-08T14:46:38Z,2021-01-08,2021-01-08 14:46:38+00:00,"The 75,000,000 great American Patriots who vot..."


In [71]:
tweets_df.to_csv(r"C:\Users\02_Florian_Benutzer\Desktop\trump_oil_analysis_v2\00_data\tweets_v3.csv", sep=",", index=False, encoding="utf-8")

### 5. Models

In [14]:
embed_model = SentenceTransformer(
    SENTENCE_MODEL
)

finbert = pipeline(
    "text-classification",
    model="ProsusAI/finbert",
    return_all_scores=True,
    top_k=None
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

### 7. FinBERT

In [75]:
def extract_finbert_features(text):

    try:

        scores = finbert(
            text,
            truncation=True,
            max_length=512
        )[0]

        scores = {
            s["label"].lower(): s["score"]
            for s in scores
        }

        pos = scores.get("positive", 0)
        neu = scores.get("neutral", 0)
        neg = scores.get("negative", 0)

        # kleine numerische Stabilität
        eps = 1e-10

        entropy = (
            -(pos * np.log(pos + eps)
              + neu * np.log(neu + eps)
              + neg * np.log(neg + eps))
        )

        confidence = max(pos, neu, neg)

        return pd.Series({

            # Original probabilities
            "finbert_positive": pos,
            "finbert_neutral": neu,
            "finbert_negative": neg,

            # Sentiment signal
            "finbert_sentiment_score": pos - neg,

            # Uncertainty / structure
            "finbert_entropy": entropy,
            "finbert_confidence": confidence,

            # your original risk/uncertainty definitions
            "financial_uncertainty_score": 1 - confidence,
            "financial_risk_sentiment": neg,
            "finbert_polarization": abs(pos - neg)
        })

    except Exception as e:
        print(e)

        return pd.Series({

            "finbert_positive": np.nan,
            "finbert_neutral": np.nan,
            "finbert_negative": np.nan,

            "finbert_sentiment_score": np.nan,
            "finbert_entropy": np.nan,
            "finbert_confidence": np.nan,

            "financial_uncertainty_score": np.nan,
            "financial_risk_sentiment": np.nan,
            "finbert_polarization": np.nan
        })

In [76]:
# tqdm.pandas()

tweets_df = pd.concat(
    [
        tweets_df,
        tweets_df["clean_text"]
        .apply(extract_finbert_features)
    ],
    axis=1
)

In [77]:
tweets_df.columns

Index(['id', 'text', 'datetime', 'date', 'timestamp_utc', 'clean_text',
       'embedding', 'finbert_positive', 'finbert_neutral', 'finbert_negative',
       'finbert_sentiment_score', 'finbert_entropy', 'finbert_confidence',
       'financial_uncertainty_score', 'financial_risk_sentiment',
       'finbert_polarization'],
      dtype='str')

In [78]:
tweets_df

,id,text,datetime,date,timestamp_utc,clean_text,embedding,finbert_positive,finbert_neutral,finbert_negative,finbert_sentiment_score,finbert_entropy,finbert_confidence,financial_uncertainty_score,financial_risk_sentiment,finbert_polarization
0,1.698309e+09,Be sure to tune in and watch Donald Trump on L...,2009-05-04T18:54:25Z,2009-05-04,2009-05-04 18:54:25+00:00,Be sure to tune in and watch Donald Trump on L...,"[0.011964291, -0.079644024, 0.05680833, -0.075...",0.061971,0.921148,0.016881,0.045090,0.316907,0.921148,0.078852,0.016881,0.045090
1,1.701461e+09,Donald Trump will be appearing on The View tom...,2009-05-05T01:00:10Z,2009-05-05,2009-05-05 01:00:10+00:00,Donald Trump will be appearing on The View tom...,"[-0.032577187, -0.03890636, 0.028219147, -0.09...",0.057433,0.923727,0.018840,0.038593,0.312210,0.923727,0.076273,0.018840,0.038593
2,1.737480e+09,Donald Trump reads Top Ten Financial Tips on L...,2009-05-08T13:38:08Z,2009-05-08,2009-05-08 13:38:08+00:00,Donald Trump reads Top Ten Financial Tips on L...,"[0.0025873815, 0.010750602, 0.020505425, -0.03...",0.129506,0.847606,0.022888,0.106617,0.491309,0.847606,0.152394,0.022888,0.106617
3,1.741161e+09,New Blog Post: Celebrity Apprentice Finale and...,2009-05-08T20:40:15Z,2009-05-08,2009-05-08 20:40:15+00:00,New Blog Post: Celebrity Apprentice Finale and...,"[-0.05556476, -0.02245654, 0.05592733, -0.0323...",0.095980,0.884520,0.019500,0.076480,0.410259,0.884520,0.115480,0.019500,0.076480
4,1.773561e+09,"""""""My persona will never be that of a wallflow...",2009-05-12T14:07:28Z,2009-05-12,2009-05-12 14:07:28+00:00,"""""""My persona will never be that of a wallflow...","[-0.0013607247, 0.08114867, 0.06519941, -0.040...",0.037306,0.935491,0.027203,0.010103,0.283117,0.935491,0.064509,0.027203,0.010103
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46687,1.346904e+18,Please support our Capitol Police and Law Enfo...,2021-01-06T19:38:58Z,2021-01-06,2021-01-06 19:38:58+00:00,Please support our Capitol Police and Law Enfo...,"[0.029277686, 0.008781331, 0.0077756834, -0.04...",0.126422,0.846731,0.026848,0.099574,0.499453,0.846731,0.153269,0.026848,0.099574
46688,1.346913e+18,I am asking for everyone at the U.S. Capitol t...,2021-01-06T20:13:26Z,2021-01-06,2021-01-06 20:13:26+00:00,I am asking for everyone at the U.S. Capitol t...,"[0.13201904, 0.02575036, 0.03230249, 0.0044190...",0.160768,0.815956,0.023276,0.137492,0.547337,0.815956,0.184044,0.023276,0.137492
46690,1.346955e+18,These are the things and events that happen wh...,2021-01-06T23:01:04Z,2021-01-06,2021-01-06 23:01:04+00:00,These are the things and events that happen wh...,"[0.014258816, 0.054777205, 0.087958075, -0.035...",0.056015,0.789560,0.154425,-0.098409,0.636474,0.789560,0.210440,0.154425,0.098409
46692,1.347555e+18,"The 75,000,000 great American Patriots who vot...",2021-01-08T14:46:38Z,2021-01-08,2021-01-08 14:46:38+00:00,"The 75,000,000 great American Patriots who vot...","[-0.016277699, -0.050014533, 0.024867747, -0.1...",0.065348,0.899488,0.035164,0.030184,0.391274,0.899488,0.100512,0.035164,0.030184


In [81]:
tweets_df.to_csv(r"C:\Users\02_Florian_Benutzer\Desktop\trump_oil_analysis_v2\00_data\tweets_v4.csv", sep=",", index=False, encoding="utf-8")

In [20]:
tweets_df = pd.read_csv(r"C:\Users\02_Florian_Benutzer\Desktop\trump_oil_analysis_v2\00_data\tweets_v4.csv")

In [21]:
tweets_df[TIMESTAMP_COL] = pd.to_datetime(
    tweets_df[TIMESTAMP_COL],
    utc=True
)

tweets_df = tweets_df.sort_values(
    TIMESTAMP_COL
).reset_index(drop=True)

In [22]:
tweets_df.dtypes

id                                         float64
text                                           str
datetime                                       str
date                                           str
timestamp_utc                  datetime64[us, UTC]
clean_text                                     str
embedding                                      str
finbert_positive                           float64
finbert_neutral                            float64
finbert_negative                           float64
finbert_sentiment_score                    float64
finbert_entropy                            float64
finbert_confidence                         float64
financial_uncertainty_score                float64
financial_risk_sentiment                   float64
finbert_polarization                       float64
dtype: object

### 8. Linguistic Features

In [23]:
import numpy as np
import pandas as pd

TEXT_COL = "clean_text"  # dein bestehender Spaltenname


# -----------------------------
# 1. Safe Caps Ratio
# -----------------------------
def caps_ratio(text):
    if pd.isna(text):
        return 0.0

    text = str(text)
    words = text.split()

    if len(words) == 0:
        return 0.0

    caps = sum(w.isupper() for w in words)
    return caps / len(words)


tweets_df["caps_ratio"] = tweets_df[TEXT_COL].apply(caps_ratio)


# -----------------------------
# 2. Exclamation Count (robust + log transform)
# -----------------------------
tweets_df["exclamation_count"] = (
    tweets_df[TEXT_COL]
    .fillna("")
    .astype(str)
    .str.count("!")
)

# stabiler für heavy tails
tweets_df["exclamation_count_log"] = np.log1p(
    tweets_df["exclamation_count"]
)


# -----------------------------
# 3. Tweet Length (konsistent + log)
# -----------------------------
tweets_df["tweet_length"] = (
    tweets_df[TEXT_COL]
    .fillna("")
    .astype(str)
    .str.split()
    .apply(len)
)

tweets_df["tweet_length_log"] = np.log1p(
    tweets_df["tweet_length"]
)


# -----------------------------
# 4. Improved Market Aggression Index
#    (robuster, weniger Skalenbias)
# -----------------------------
tweets_df["market_aggression_index"] = (
    0.6 * tweets_df["caps_ratio"] +
    0.2 * tweets_df["exclamation_count_log"] +
    0.2 * tweets_df["tweet_length_log"]
)

### 9. Temporal Features

In [25]:
# tweets_df = tweets_df.sort_values("timestamp").copy()

tweets_df["time_since_last_tweet_min"] = (
    tweets_df["timestamp_utc"].diff().dt.total_seconds() / 60
)

tweets_df["time_since_last_tweet_min"] = (
    tweets_df["time_since_last_tweet_min"].fillna(0)
)

### 10. Leakage-Safe Rolling Features

In [26]:
# tweets_df = tweets_df.copy()

tweets_df[TIMESTAMP_COL] = pd.to_datetime(tweets_df[TIMESTAMP_COL])
tweets_df = tweets_df.sort_values(TIMESTAMP_COL)

tweets_df = tweets_df.set_index(TIMESTAMP_COL)

tweets_df["rolling_tweet_frequency_60m"] = (
    pd.Series(
        1,
        index=tweets_df.index
    )
    .rolling("60min")
    .sum()
    .shift(1)
    .fillna(0)
)

tweets_df["rolling_tweet_frequency_6h"] = (
    pd.Series(
        1,
        index=tweets_df.index
    )
    .rolling("6h")
    .sum()
    .shift(1)
    .fillna(0)
)

tweets_df["rolling_tweet_frequency_24h"] = (
    pd.Series(
        1,
        index=tweets_df.index
    )
    .rolling("24h")
    .sum()
    .shift(1)
    .fillna(0)
)

tweets_df["tweet_burst_indicator"] = (
    tweets_df["rolling_tweet_frequency_60m"] > 3
).astype(int)

tweets_df["tweet_acceleration_6h"] = (
    tweets_df["rolling_tweet_frequency_6h"]
    - tweets_df["rolling_tweet_frequency_60m"]
)

### 11. Sentiment History

In [27]:
tweets_df["sentiment_delta_vs_previous"] = (

    tweets_df[
        "finbert_sentiment_score"
    ]
    -
    tweets_df[
        "finbert_sentiment_score"
    ].shift(1)

)

tweets_df["rolling_sentiment_mean_60m"] = (

    tweets_df[
        "finbert_sentiment_score"
    ]
    .rolling("60min")
    .mean()
    .shift(1)

)

tweets_df["rolling_sentiment_std_60m"] = (

    tweets_df[
        "finbert_sentiment_score"
    ]
    .rolling("60min")
    .std()
    .shift(1)

)

In [28]:
tweets_df = tweets_df.reset_index()

In [29]:
tweets_df.isnull().sum()

timestamp_utc                      0
id                                 0
text                               0
datetime                           0
date                               0
clean_text                         0
embedding                          0
finbert_positive                   0
finbert_neutral                    0
finbert_negative                   0
finbert_sentiment_score            0
finbert_entropy                    0
finbert_confidence                 0
financial_uncertainty_score        0
financial_risk_sentiment           0
finbert_polarization               0
caps_ratio                         0
exclamation_count                  0
exclamation_count_log              0
tweet_length                       0
tweet_length_log                   0
market_aggression_index            0
time_since_last_tweet_min          0
rolling_tweet_frequency_60m        0
rolling_tweet_frequency_6h         0
rolling_tweet_frequency_24h        0
tweet_burst_indicator              0
t

In [30]:
tweets_df = tweets_df.fillna(0)

In [31]:
tweets_df.isna().sum()

timestamp_utc                  0
id                             0
text                           0
datetime                       0
date                           0
clean_text                     0
embedding                      0
finbert_positive               0
finbert_neutral                0
finbert_negative               0
finbert_sentiment_score        0
finbert_entropy                0
finbert_confidence             0
financial_uncertainty_score    0
financial_risk_sentiment       0
finbert_polarization           0
caps_ratio                     0
exclamation_count              0
exclamation_count_log          0
tweet_length                   0
tweet_length_log               0
market_aggression_index        0
time_since_last_tweet_min      0
rolling_tweet_frequency_60m    0
rolling_tweet_frequency_6h     0
rolling_tweet_frequency_24h    0
tweet_burst_indicator          0
tweet_acceleration_6h          0
sentiment_delta_vs_previous    0
rolling_sentiment_mean_60m     0
rolling_se

In [ ]:
# tweets_df.to_csv(r"C:\Users\02_Florian_Benutzer\Desktop\trump_oil_analysis_v2\00_data\tweets_v5.csv", sep=",", index=False, encoding="utf-8")

### 12. In Office

Wir betrachten nur die Tweets der ersten Amtszeit

In [32]:
def add_is_president_feature(df, timestamp_col="timestamp_utc"):
    df[timestamp_col] = pd.to_datetime(df[timestamp_col], utc=True)
    
    start = pd.Timestamp("2017-01-20", tz="UTC")
    end   = pd.Timestamp("2021-01-20", tz="UTC")
    
    df["is_trump_president"] = (
        (df[timestamp_col] >= start) & 
        (df[timestamp_col] < end)
    ).astype(int)
    
    return df

In [33]:
tweets_df = add_is_president_feature(tweets_df)

In [34]:
tweets_df = tweets_df[tweets_df["is_trump_president"] == 1]

In [41]:
tweets_df.dtypes

timestamp_utc                  datetime64[us, UTC]
id                                         float64
text                                           str
datetime                                       str
date                                           str
clean_text                                     str
embedding                                      str
finbert_positive                           float64
finbert_neutral                            float64
finbert_negative                           float64
finbert_sentiment_score                    float64
finbert_entropy                            float64
finbert_confidence                         float64
financial_uncertainty_score                float64
financial_risk_sentiment                   float64
finbert_polarization                       float64
caps_ratio                                 float64
exclamation_count                            int64
exclamation_count_log                      float64
tweet_length                   

In [42]:
tweets_df.drop(columns=["id", "text", "datetime", "date"], inplace=True)

In [46]:
tweets_df.columns

Index(['timestamp_utc', 'clean_text', 'embedding', 'finbert_positive',
       'finbert_neutral', 'finbert_negative', 'finbert_sentiment_score',
       'finbert_entropy', 'finbert_confidence', 'financial_uncertainty_score',
       'financial_risk_sentiment', 'finbert_polarization', 'caps_ratio',
       'exclamation_count', 'exclamation_count_log', 'tweet_length',
       'tweet_length_log', 'market_aggression_index',
       'time_since_last_tweet_min', 'rolling_tweet_frequency_60m',
       'rolling_tweet_frequency_6h', 'rolling_tweet_frequency_24h',
       'tweet_burst_indicator', 'tweet_acceleration_6h',
       'sentiment_delta_vs_previous', 'rolling_sentiment_mean_60m',
       'rolling_sentiment_std_60m', 'is_trump_president'],
      dtype='str')

In [47]:
# tweets_df.to_csv(r"C:\Users\02_Florian_Benutzer\Desktop\trump_oil_analysis_v2\00_data\tweets_v6_inOffice.csv", sep=",", index=False, encoding="utf-8")
tweets_df.to_parquet(r"C:\Users\02_Florian_Benutzer\Desktop\trump_oil_analysis_v2\00_data\tweets_v5.parquet", index=False)

### 8. Semantic Centroids

In [ ]:
# reference_texts = {

#     "wti":
#     "oil supply shock geopolitics OPEC crude war sanctions energy inflation",

#     "geo":
#     "war sanctions Russia Iran Middle East conflict military escalation",

#     "energy":
#     "oil production OPEC drilling supply demand crude inventory",

#     "macro":
#     "inflation interest rates GDP recession monetary policy Fed",

#     "policy":
#     "policy decision executive order sanctions trade war"
# }

# reference_embeddings = {

#     name: embed_model.encode(
#         text,
#         normalize_embeddings=True
#     )

#     for name, text
#     in reference_texts.items()
# }

### 9. Similarity Features

In [ ]:
# def similarity_to_reference(
#     emb,
#     ref
# ):

#     return cosine_similarity(
#         emb.reshape(1, -1),
#         ref.reshape(1, -1)
#     )[0, 0]


# for name, ref in reference_embeddings.items():

#     tweets_df[f"{name}_similarity"] = [

#         similarity_to_reference(
#             emb,
#             ref
#         )

#         for emb in embeddings
#     ]


# tweets_df.rename(
#     columns={
#         "wti_similarity":
#             "wti_relevance_score",

#         "geo_similarity":
#             "geopolitical_similarity_score",

#         "energy_similarity":
#             "energy_supply_similarity_score",

#         "macro_similarity":
#             "macro_economy_similarity_score",

#         "policy_similarity":
#             "policy_strength_score"
#     },
#     inplace=True
# )

# Beispiel wie man unten definierte Centroid Berechnung in einer Cross Validation Pipeline durchführen kann

Verbesserte Version von ChatGPT für Schritt 8 und 9

Du baust ein Zero-Shot Topic-Scoring System:

- Du definierst Beispieltexte pro Topic
- Du erzeugst daraus Embedding-Zentren (Centroids)
- Du misst Cosine Similarity zwischen Tweet-Embeddings und Topic-Zentroiden
- Du speicherst diese Scores als Features im DataFrame

In [23]:
# tweets_df.to_csv(r"C:\Users\02_Florian_Benutzer\Desktop\trump_oil_analysis_v2\00_data\tweets_v6_inOffice.csv", sep=",", index=False, encoding="utf-8")
tweets_df = pd.read_parquet(r"C:\Users\02_Florian_Benutzer\Desktop\trump_oil_analysis_v2\00_data\tweets_v5.parquet")

In [24]:
tweets_df.isna().sum()

timestamp_utc                  0
clean_text                     0
embedding                      0
finbert_positive               0
finbert_neutral                0
finbert_negative               0
finbert_sentiment_score        0
finbert_entropy                0
finbert_confidence             0
financial_uncertainty_score    0
financial_risk_sentiment       0
finbert_polarization           0
caps_ratio                     0
exclamation_count              0
exclamation_count_log          0
tweet_length                   0
tweet_length_log               0
market_aggression_index        0
time_since_last_tweet_min      0
rolling_tweet_frequency_60m    0
rolling_tweet_frequency_6h     0
rolling_tweet_frequency_24h    0
tweet_burst_indicator          0
tweet_acceleration_6h          0
sentiment_delta_vs_previous    0
rolling_sentiment_mean_60m     0
rolling_sentiment_std_60m      0
is_trump_president             0
dtype: int64

In [25]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity


topic_examples = {

    # ============================================================
    # 1. DIRECT WTI PRICE SIGNALS (DIRECTIONAL)
    # ============================================================

    "wti_bullish": [
        "oil supply shortage",
        "crude oil supply disruption",
        "OPEC cuts production",
        "unexpected production decline",
        "strategic petroleum reserve drained",
        "refinery outages reduce supply",
        "pipeline disruption in major producer",
        "sanctions reduce oil exports",
        "crude inventory drawdown",
        "energy market tightening rapidly",
        "oil price shock due to supply shock"
    ],

    "wti_bearish": [
        "oil production increase",
        "OPEC increases output",
        "strategic petroleum reserve release",
        "weak global oil demand",
        "economic slowdown reduces fuel consumption",
        "record high oil inventories",
        "shale oil production surge",
        "fuel demand collapse",
        "oversupply in oil market",
        "recession fears reduce energy demand"
    ],


    # ============================================================
    # 2. OIL MARKET MECHANISMS (STRUCTURAL DRIVERS)
    # ============================================================

    "energy_supply": [
        "OPEC production decision",
        "crude oil inventories report",
        "oil drilling activity increases",
        "energy supply constraints",
        "refinery utilization rates change",
        "pipeline disruption",
        "global energy demand shifts",
        "oil export volumes fluctuate",
        "supply chain disruptions in energy markets",
        "production quota changes"
    ],

    "geopolitical_oil_risk": [
        "Middle East conflict escalation",
        "Strait of Hormuz tensions",
        "Iran sanctions on oil exports",
        "Russia oil export restrictions",
        "war affects global energy supply",
        "pipeline attacks in oil region",
        "Saudi production risk",
        "OPEC instability",
        "international energy embargo",
        "geopolitical instability affects oil markets"
    ],


    # ============================================================
    # 3. MACRO / USD / FED CHANNEL
    # ============================================================

    "usd_strength_pressure": [
        "strong US dollar",
        "USD index rises",
        "dollar rally impacts commodities",
        "currency tightening",
        "exchange rate pressure on oil",
        "global dollar liquidity tightening"
    ],

    "fed_monetary_policy": [
        "Federal Reserve interest rate hike",
        "hawkish Fed stance",
        "monetary tightening policy",
        "inflation concerns rise",
        "economic recession fears",
        "central bank tightening cycle",
        "financial conditions tighten"
    ],

    "risk_sentiment": [
        "risk-off sentiment increases",
        "global market uncertainty rises",
        "equity market selloff",
        "flight to safety assets",
        "financial instability concerns",
        "market panic increases volatility"
    ],


    # ============================================================
    # 4. TRUMP-SPECIFIC LANGUAGE SIGNALS
    # ============================================================

    "trump_energy_policy": [
        "drill baby drill",
        "energy independence policy",
        "US oil dominance strategy",
        "expand domestic oil drilling",
        "boost fossil fuel production",
        "reduce energy regulations",
        "open federal lands for drilling",
        "pipeline approval acceleration",
        "increase US oil production",
        "America first energy policy"
    ],

    "trump_market_shock_language": [
        "fake news",
        "disaster situation",
        "total failure",
        "huge crisis",
        "historic win",
        "biggest ever seen",
        "strongest economy ever",
        "breaking news announcement",
        "never seen before situation",
        "massive policy change"
    ],


    # ============================================================
    # 5. POLICY / SURPRISE / EVENT SHOCK
    # ============================================================

    "policy_shock": [
        "unexpected policy announcement",
        "surprise tariff decision",
        "unanticipated sanctions",
        "shock policy statement",
        "market not expecting decision",
        "sudden trade policy change",
        "geopolitical surprise announcement",
        "unexpected government intervention",
        "regulatory shock event",
        "breaking policy shift"
    ]
}

Centroids berechnen

In [26]:
# ============================================================
# 2. Topic-Centroids erstellen
# ============================================================

def build_centroid(texts, embed_model):

    embeddings = embed_model.encode(
        texts,
        normalize_embeddings=True
    )

    centroid = np.mean(
        embeddings,
        axis=0
    )

    centroid = centroid / np.linalg.norm(
        centroid
    )

    return centroid


topic_centroids = {

    topic: build_centroid(
        examples,
        embed_model
    )

    for topic, examples
    in topic_examples.items()
}

Tweet Embedding berechnen - wude weiter oben schon gemacht daher auskommentiert

### Embeddings

In [ ]:
embeddings = embed_model.encode(
    tweets_df["clean_text"].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True
)

embeddings = np.asarray(embeddings)

tweets_df["embedding"] = list(
    embeddings
)

Batches:   0%|          | 0/477 [00:00<?, ?it/s]

In [ ]:
# tweet_texts = tweets_df["tweet_text"].tolist()

# tweet_embeddings = embed_model.encode(
#     tweet_texts,
#     normalize_embeddings=True,
#     show_progress_bar=True
# )

In [27]:
embeddings

array([[ 0.0144826 ,  0.02638815,  0.0550416 , ...,  0.04797632,
        -0.09664363,  0.00838226],
       [-0.02392054,  0.08108884,  0.10887722, ...,  0.07767763,
        -0.0194047 ,  0.01249293],
       [-0.04619306, -0.0101754 ,  0.04101638, ..., -0.03580762,
        -0.05680258, -0.01289774],
       ...,
       [ 0.01425882,  0.0547772 ,  0.08795808, ...,  0.00248005,
         0.01035586, -0.00417345],
       [-0.0162777 , -0.05001453,  0.02486775, ..., -0.01108151,
         0.03540413, -0.02160552],
       [ 0.00582693,  0.03654156,  0.0494696 , ..., -0.03159347,
        -0.03307797,  0.00380668]], shape=(15242, 384), dtype=float32)

Similarities vektorisiert berechnen

In [28]:
# ============================================================
# 3. Similarity Features erzeugen
# ============================================================

for topic, centroid in topic_centroids.items():

    scores = cosine_similarity(
        embeddings,
        centroid.reshape(1, -1)
    ).flatten()

    tweets_df[f"{topic}_score"] = scores

Ergebnis

In [ ]:
[
    "wti_bullish_score",
    "wti_bearish_score",
    "energy_supply_score",
    "geopolitical_oil_risk_score",
    "usd_strength_pressure_score",
    "fed_monetary_policy_score",
    "risk_sentiment_score",
    "trump_energy_policy_score",
    "trump_market_shock_language_score",
    "policy_shock_score"
]

### Optional: stärkere Features für XGBoost

Dominantes Topic

In [29]:
topic_cols = [
    "wti_bullish_score",
    "wti_bearish_score",
    "energy_supply_score",
    "geopolitical_oil_risk_score",
    "usd_strength_pressure_score",
    "fed_monetary_policy_score",
    "risk_sentiment_score",
    "trump_energy_policy_score",
    "trump_market_shock_language_score",
    "policy_shock_score"
    # "wti_relevance_score",
    # "geopolitical_score",
    # "energy_supply_score",
    # "macro_economy_score",
    # "policy_score"
]

tweets_df["dominant_topic"] = (
    tweets_df[topic_cols]
    .idxmax(axis=1)
)

Für XGBoost anschließend One-Hot Encoding

In [47]:
tweets_df["dominant_topic"] = (
    tweets_df["dominant_topic"]
    .str.replace("_score", "", regex=False)
)

dominant_topic_ohe = pd.get_dummies(
    tweets_df["dominant_topic"],
    prefix="topic"
).astype(int)

In [48]:
tweets_df = pd.concat(
    [tweets_df, dominant_topic_ohe],
    axis=1
)

Topic Concentration - Misst, ob ein Tweet sehr fokussiert ist.

In [30]:
tweets_df["topic_concentration"] = (
    tweets_df[topic_cols]
    .max(axis=1)
    -
    tweets_df[topic_cols]
    .mean(axis=1)
)

Relevanzfilter - Viele Tweets werden nichts mit Öl zu tun haben

In [52]:
# tweets_df.to_csv(r"C:\Users\02_Florian_Benutzer\Desktop\trump_oil_analysis_v2\00_data\tweets_v6_inOffice.csv", sep=",", index=False, encoding="utf-8")
tweets_df.to_parquet(r"C:\Users\02_Florian_Benutzer\Desktop\trump_oil_analysis_v2\00_data\tweets_v6.parquet", index=False)

In [ ]:
# tweets_df = pd.read_parquet(r"C:\Users\02_Florian_Benutzer\Desktop\trump_oil_analysis_v2\00_data\tweets_v6.parquet")

### 14. Novelty

In [55]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity


def compute_semantic_features(embeddings, window=50, use_percentile=95):
    """
    Computes:
    1. global novelty
    2. local semantic change
    3. rolling novelty (window-based, leakage-safe)

    Parameters
    ----------
    embeddings : np.ndarray (n_samples, dim)
    window : int
        rolling window size
    use_percentile : float
        percentile used instead of max for robustness

    Returns
    -------
    dict of np.ndarray
    """

    embeddings = np.asarray(embeddings)
    n = len(embeddings)

    global_novelty = np.zeros(n)
    local_change = np.zeros(n)
    rolling_novelty = np.zeros(n)

    # ---- 1. GLOBAL NOVELTY ----
    for i in range(1, n):
        sims = cosine_similarity(
            embeddings[i].reshape(1, -1),
            embeddings[:i]
        )[0]

        # robust version instead of pure max
        sim = np.percentile(sims, use_percentile)
        global_novelty[i] = 1 - sim


    # ---- 2. LOCAL CHANGE ----
    for i in range(1, n):
        local_change[i] = 1 - cosine_similarity(
            embeddings[i].reshape(1, -1),
            embeddings[i - 1].reshape(1, -1)
        )[0, 0]


    # ---- 3. ROLLING NOVELTY ----
    for i in range(1, n):
        start = max(0, i - window)

        if start == i:
            continue

        sims = cosine_similarity(
            embeddings[i].reshape(1, -1),
            embeddings[start:i]
        )[0]

        # more stable than max
        sim = np.percentile(sims, use_percentile)
        rolling_novelty[i] = 1 - sim


    return {
        "semantic_global_novelty": global_novelty,
        "semantic_local_change": local_change,
        "semantic_rolling_novelty": rolling_novelty
    }

Anwendung in Dataframe

In [56]:
features = compute_semantic_features(embeddings, window=50)

tweets_df["semantic_global_novelty"] = features["semantic_global_novelty"]
tweets_df["semantic_local_change"] = features["semantic_local_change"]
tweets_df["semantic_rolling_novelty"] = features["semantic_rolling_novelty"]

In [59]:
# tweets_df.to_csv(r"C:\Users\02_Florian_Benutzer\Desktop\trump_oil_analysis_v2\00_data\tweets_v6_inOffice.csv", sep=",", index=False, encoding="utf-8")
tweets_df.to_parquet(r"C:\Users\02_Florian_Benutzer\Desktop\trump_oil_analysis_v2\00_data\tweets_v6.parquet", index=False)

## Markierung: Hier endet das Feature Engineering und restliche Transformationen werden in dem modeling.ipynb durchgeführt

# ACHTUNG: Alle nachfolgenden Features werden Stand 14.06.2026 nicht in den Datensatz aufgenommen

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

# ----------------------------
# 1. Safety: Missing Values
# ----------------------------
features = [
    "finbert_sentiment_score",
    "geopolitical_similarity_score",
    "market_aggression_index",
    "wti_relevance_score"
]

tweets_df[features] = tweets_df[features].fillna(0)

# ----------------------------
# 2. Scaling (wichtig für stabile Interactions)
# ----------------------------
scaler = StandardScaler()

scaled_cols = [
    "finbert_sentiment_score",
    "geopolitical_similarity_score",
    "market_aggression_index",
    "wti_relevance_score"
]

tweets_df[[col + "_z" for col in scaled_cols]] = scaler.fit_transform(
    tweets_df[scaled_cols]
)

# ----------------------------
# 3. Robust Interaction Features
# ----------------------------

eps = 1e-9  # numerical stability

# (A) Multiplicative interactions (baseline, but stabilized)
tweets_df["sentiment_x_geopolitics"] = (
    tweets_df["finbert_sentiment_score_z"] *
    tweets_df["geopolitical_similarity_score_z"]
)

tweets_df["arousal_x_oil_relevance"] = (
    tweets_df["market_aggression_index_z"] *
    tweets_df["wti_relevance_score_z"]
)

# (B) Log-stabilized interactions (often better in finance NLP)
tweets_df["sentiment_x_geopolitics_log"] = (
    np.log1p(np.abs(tweets_df["finbert_sentiment_score_z"])) *
    np.log1p(np.abs(tweets_df["geopolitical_similarity_score_z"]))
)

tweets_df["arousal_x_oil_relevance_log"] = (
    np.log1p(np.abs(tweets_df["market_aggression_index_z"])) *
    np.log1p(np.abs(tweets_df["wti_relevance_score_z"]))
)

# ----------------------------
# 4. Optional: Directional interaction (sign-preserving)
# ----------------------------
tweets_df["sentiment_geopolitics_signed"] = (
    tweets_df["finbert_sentiment_score_z"] *
    tweets_df["geopolitical_similarity_score_z"] *
    np.sign(tweets_df["finbert_sentiment_score_z"])
)

# ----------------------------
# 5. Sanity check
# ----------------------------
interaction_cols = [
    "sentiment_x_geopolitics",
    "arousal_x_oil_relevance",
    "sentiment_x_geopolitics_log",
    "arousal_x_oil_relevance_log"
]

print(tweets_df[interaction_cols].describe())

In [ ]:
# tweets_df["sentiment_x_geopolitics"] = (

#     tweets_df[
#         "finbert_sentiment_score"
#     ]
#     *
#     tweets_df[
#         "geopolitical_similarity_score"
#     ]
# )

# tweets_df["arousal_x_oil_relevance"] = (

#     tweets_df[
#         "market_aggression_index"
#     ]
#     *
#     tweets_df[
#         "wti_relevance_score"
#     ]
# )

### 17. Market Shock Index

In [ ]:
tweets_df["tweet_market_shock_index"] = (

      0.25 * tweets_df["finbert_sentiment_score"]
    + 0.20 * tweets_df["market_aggression_index"]
    + 0.20 * tweets_df["geopolitical_similarity_score"]
    + 0.15 * tweets_df["semantic_rolling_novelty"]
    + 0.20 * tweets_df["wti_relevance_score"]
)

### 18. Time Split

In [ ]:
# empfohlen

split_time = tweets_df[TIMESTAMP_COL].sort_values().iloc[int(len(tweets_df) * (1 - TEST_SIZE))]

# oder

split_time = tweets_df[TIMESTAMP_COL].quantile(1 - TEST_SIZE, interpolation="linear")

Step 1: Split Zuerst

In [ ]:
tweets_df = tweets_df.sort_values(TIMESTAMP_COL).reset_index(drop=True)

split_time = tweets_df[TIMESTAMP_COL].quantile(1 - TEST_SIZE)

train_df = tweets_df[tweets_df[TIMESTAMP_COL] <= split_time].copy()
test_df  = tweets_df[tweets_df[TIMESTAMP_COL] > split_time].copy()

Step 2: Scaling nur auf Train fitten

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

scale_cols = [
    "finbert_sentiment_score",
    "geopolitical_similarity_score",
    "market_aggression_index",
    "wti_relevance_score"
]

train_df[[c + "_z" for c in scale_cols]] = scaler.fit_transform(train_df[scale_cols])
test_df[[c + "_z" for c in scale_cols]] = scaler.transform(test_df[scale_cols])

Step 3: Feature Engineering separat

In [ ]:
def create_features(df):
    eps = 1e-9
    
    df["sentiment_x_geopolitics"] = (
        df["finbert_sentiment_score_z"] *
        df["geopolitical_similarity_score_z"]
    )

    df["arousal_x_oil_relevance"] = (
        df["market_aggression_index_z"] *
        df["wti_relevance_score_z"]
    )

    return df


train_df = create_features(train_df)
test_df = create_features(test_df)

Alternativer Ansatz

In [29]:
tweets_df = tweets_df.reset_index()

split_time = tweets_df[
    TIMESTAMP_COL
].quantile(
    1 - TEST_SIZE
)

train_df = tweets_df[
    tweets_df[TIMESTAMP_COL]
    <= split_time
].copy()

test_df = tweets_df[
    tweets_df[TIMESTAMP_COL]
    > split_time
].copy()

TypeError: unsupported operand type(s) for -: 'str' and 'str'

### 19. PCA (Leakage Safe)

In [ ]:
train_embeddings = np.vstack(
    train_df["embedding"]
)

test_embeddings = np.vstack(
    test_df["embedding"]
)

scaler = StandardScaler()

train_embeddings_scaled = (
    scaler.fit_transform(
        train_embeddings
    )
)

test_embeddings_scaled = (
    scaler.transform(
        test_embeddings
    )
)

pca = PCA(
    n_components=PCA_COMPONENTS,
    random_state=42
)

train_pca = pca.fit_transform(
    train_embeddings_scaled
)

test_pca = pca.transform(
    test_embeddings_scaled
)

### 20. PCA Features

In [ ]:
for i in range(
    PCA_COMPONENTS
):

    train_df[
        f"embedding_pca_{i}"
    ] = train_pca[:, i]

    test_df[
        f"embedding_pca_{i}"
    ] = test_pca[:, i]

### 21. BERTopic

In [ ]:
topic_model = BERTopic(
    verbose=True
)

train_topics, _ = (
    topic_model.fit_transform(
        train_df["clean_text"]
        .tolist()
    )
)

test_topics, _ = (
    topic_model.transform(
        test_df["clean_text"]
        .tolist()
    )
)

train_df["topic_id"] = (
    train_topics
)

test_df["topic_id"] = (
    test_topics
)

### 22. KMeans

In [ ]:
kmeans = KMeans(
    n_clusters=N_CLUSTERS,
    random_state=42,
    n_init="auto"
)

kmeans.fit(train_pca)

train_df["cluster_id"] = (
    kmeans.predict(train_pca)
)

test_df["cluster_id"] = (
    kmeans.predict(test_pca)
)

### 23. Final Cleanup

In [ ]:
final_df = pd.concat(
    [train_df, test_df]
)

final_df = final_df.sort_values(
    TIMESTAMP_COL
)

final_df = final_df.replace(
    [np.inf, -np.inf],
    np.nan
)

final_df = final_df.fillna(0)

### 24. Export

In [ ]:
final_df.to_parquet(
    "tweet_feature_store.parquet",
    index=False
)

final_df.to_csv(
    "tweet_feature_store.csv",
    index=False
)